# Notebook 06 — No-retrain improvement probe on Baseline-01
Solar Filament Segmentation Challenge 2026 — Kaggle edition

Notebook 05 showed three things about Baseline-01 (best checkpoint, validation PQ 0.2881):
1. **BatchNorm running statistics are unstable at some epochs** (epoch-40 snapshot: PQ 0.000 -> 0.270 after re-estimating BN statistics).
2. **Small predicted objects are almost all false positives** (predictions <200 px: 399 of 401 are FP; the ground truth has essentially no objects that small).
3. Closing small gaps between fragments helps a little, and the best sweep settings sat at the **edge of the grid**.

This notebook tests, with **no training at all**, how much of that can be recovered at inference time:
* **A. BN re-estimation**, **B. flip test-time augmentation (TTA)**, **C. snapshot ensemble** (epochs best/30/40/50 averaged)
* **D. extended post-processing sweep** (threshold, minimum object size, hole-closing) with the split-half protocol
* **E. Pooled-annotator ground truth**: how does PQ change if ALL annotation records of an image count as ground truth?
  (Which of "primary record only" or "all records" the leaderboard uses is something to confirm from the competition's evaluation page.)

**Setup:** Add Input -> the competition AND the **finished** Baseline-01 FULL version output (must contain `checkpoint_epoch_0X0.pth`). GPU on. Save & Run All.

Everything here is a diagnostic on the validation set that also selected the checkpoint; the split-half protocol limits, but does not remove, optimism.

In [1]:
# ============================================================
# CONFIGURATION — the only cell you should need to edit
# ============================================================

# Competition data: attach it via "Add Input". Path is auto-detected (notebooks 1 and 2 used the first one).
KAGGLE_DATA_CANDIDATES = [
    "/kaggle/input/competitions/filament-segmentation-2026",
    "/kaggle/input/filament-segmentation-2026",
]

RUN_MODE = "FULL"               # <<< REAL Baseline-01.  ("SMOKE" | "PILOT" | "FULL", see the markdown cell above)
assert RUN_MODE in ("SMOKE", "PILOT", "FULL"), "RUN_MODE must be SMOKE, PILOT or FULL"
DEBUG_MODE = (RUN_MODE == "SMOKE")   # reduced-image-count shortcuts exist ONLY in SMOKE mode

OUTPUT_DIR = "/kaggle/working"  # kept only if you "Save Version"; see markdown above

SEED = 42
RESUME = True                   # resume from latest valid checkpoint if one exists

# ---- experiment parameters (Baseline-01) — same field names as Notebook 03 ----
CFG = {
    "experiment_id": "Baseline-01" if RUN_MODE == "FULL" else f"Baseline-01-{RUN_MODE}",
    "train_resolution": 512 if DEBUG_MODE else 1024,
    "batch_size": 2 if DEBUG_MODE else 4,
    "epochs": {"SMOKE": 20, "PILOT": 3, "FULL": 50}[RUN_MODE],
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "model": "UNet-base32",
    "loss": "BCE+Dice",
    "prob_threshold": 0.5,       # single configurable baseline threshold (NOT tuned)
    "min_instance_px": 0,        # no size filtering in the baseline
    "debug_n_train": 24,         # smoke-test only
    "debug_n_val": 8,            # smoke-test only
    # ---- run-infrastructure settings (do not change the methodology) ----
    "use_amp": True,             # automatic mixed precision for training / val loss
    "num_workers": None,         # None -> min(4, cpu_count)
    "pq_eval_every": 10,         # monitor val PQ every N epochs (0 = only final). Monitoring only.
    "snapshot_every": 10,        # keep a full-state snapshot every N epochs
    "ckpt_every": 2,             # write latest_checkpoint.pth every N epochs (and always at the last epoch)
    "fsync_checkpoints": False,  # fsync was a suspected cause of slow saves on Kaggle disk; atomic rename is kept
}

print(f"RUN_MODE={RUN_MODE} (DEBUG_MODE={DEBUG_MODE}) | resolution={CFG['train_resolution']} "
      f"| epochs={CFG['epochs']} | batch={CFG['batch_size']} | AMP={CFG['use_amp']}")

RUN_MODE=FULL (DEBUG_MODE=False) | resolution=1024 | epochs=50 | batch=4 | AMP=True


In [2]:
import os, sys, json, random, re, warnings, math, shutil, time, hashlib, platform, gc, datetime, copy
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU available (no CPU fallback). Kaggle panel -> Session options -> Accelerator -> GPU.")
DEVICE = torch.device("cuda:0")
CFG["num_workers"] = min(4, os.cpu_count() or 2)
print(f"torch {torch.__version__} | {torch.cuda.get_device_name(0)} | workers {CFG['num_workers']}")

OUT = Path(OUTPUT_DIR) / "improve"; OUT.mkdir(parents=True, exist_ok=True)

# ---- locate the saved Baseline-01 FULL output (attached as an Input) ----
# Searches for the CHECKPOINT FILE anywhere under /kaggle/input (the notebook's title does not matter).
hits = sorted(Path("/kaggle/input").rglob("baseline_best.pth"))
print("baseline_best.pth found at:", [str(h) for h in hits] if hits else "NOWHERE")
if not hits:
    print("\nWhat is mounted under /kaggle/input (depth <= 3):")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root[len("/kaggle/input"):].count(os.sep)
        if depth <= 3:
            pth = [f for f in files if f.endswith(".pth")]
            print(f"  {root}  | files: {len(files)}" + (f" | .pth: {pth[:6]}" if pth else ""))
    raise FileNotFoundError(
        "No baseline_best.pth under /kaggle/input.\n"
        "The input you added contains no checkpoint files. Most likely the saved version has an EMPTY Output tab "
        "(Quick Save, still running, or failed). Open that notebook on Kaggle -> the version -> 'Output' tab: it must list "
        "baseline_best.pth, latest_checkpoint.pth, checkpoint_epoch_0X0.pth ... If it is empty, re-run the Baseline-01 FULL "
        "notebook with 'Save & Run All' and wait for the version to finish, then add THAT version's output as the input.")
PREV = max((h.parent for h in hits), key=lambda p: p.stat().st_mtime)
print("Baseline-01 output folder used:", PREV)
print("files:", sorted(p.name for p in PREV.iterdir()))

torch 2.11.0+cu128 | Tesla T4 | workers 4
baseline_best.pth found at: ['/kaggle/input/notebooks/parsabehjattabrizi/baseline-best/full/baseline_best.pth']
Baseline-01 output folder used: /kaggle/input/notebooks/parsabehjattabrizi/baseline-best/full
files: ['baseline_best.pth', 'baseline_best_meta.json', 'baseline_report.txt', 'baseline_results.csv', 'checkpoint_epoch_010.pth', 'checkpoint_epoch_020.pth', 'checkpoint_epoch_030.pth', 'checkpoint_epoch_040.pth', 'checkpoint_epoch_050.pth', 'experiment_log.csv', 'iou_dice_dist.png', 'latest_checkpoint.pth', 'latest_checkpoint_prev.pth', 'per_image_metrics.csv', 'run_config.json', 'split_metadata.csv', 'training_curve.png', 'training_history.csv', 'validation_gallery.png', 'validation_gallery_worst3.png']


In [3]:
# ============================================================
# DATA DISCOVERY on Kaggle (/kaggle/input) — same discovery logic as Notebook 03
# ============================================================

def acquire_data_root():
    for c in KAGGLE_DATA_CANDIDATES:
        if Path(c).exists():
            return Path(c)
    base = Path("/kaggle/input")
    if base.exists() and any(base.rglob("train_images")):
        print("WARNING: standard path not found; using /kaggle/input and auto-discovery.")
        return base
    raise FileNotFoundError(
        "Competition data not found under /kaggle/input.\n"
        "Fix: right-hand panel -> Add Input -> search 'filament-segmentation-2026' -> Add.")

DATA_ROOT = acquire_data_root()
print("DATA_ROOT:", DATA_ROOT)

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def discover_dataset(root):
    """Find train/test image dirs and the COCO annotation JSON. Fail loudly."""
    img_dirs = []
    for d in sorted({p.parent for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS}):
        n = sum(1 for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)
        img_dirs.append((d, n))
        print(f"  image dir: {d.relative_to(root)}  ({n} images)")
    train_cands = [d for d, _ in img_dirs
                   if "train" in str(d).lower() and "test" not in str(d).lower()]
    test_cands = [d for d, _ in img_dirs
                  if "test" in str(d).lower() and "train" not in str(d).lower()]
    if len(train_cands) != 1 or len(test_cands) != 1:
        raise RuntimeError(
            f"Ambiguous dataset layout. train candidates={train_cands}, "
            f"test candidates={test_cands}. Inspect the listing above.")
    coco = None
    for p in root.rglob("*.json"):
        try:
            with open(p) as f:
                d = json.load(f)
            if isinstance(d, dict) and "images" in d and "annotations" in d:
                coco = p; break
        except Exception:
            pass
    if coco is None:
        raise RuntimeError(f"No COCO-style annotation JSON found under {root}")
    return train_cands[0], test_cands[0], coco

TRAIN_DIR, TEST_DIR, ANN_PATH = discover_dataset(DATA_ROOT)

train_paths = sorted(p for p in TRAIN_DIR.iterdir() if p.suffix.lower() in IMG_EXTS)
test_paths  = sorted(p for p in TEST_DIR.iterdir()  if p.suffix.lower() in IMG_EXTS)
name_to_path = {p.name: p for p in train_paths}

print(f"\ntrain images: {len(train_paths)} | test images: {len(test_paths)}")
print(f"annotations: {ANN_PATH.name}")
assert len(train_paths) > 0 and len(test_paths) > 0

DATA_ROOT: /kaggle/input/competitions/filament-segmentation-2026
  image dir: MAGFiLO_1.0_Kaggle_2026/test/test_images  (180 images)
  image dir: MAGFiLO_1.0_Kaggle_2026/train/train_images  (707 images)

train images: 707 | test images: 180
annotations: MAGFiLO_1.0_Annotations_kaggle2026_train.json


In [4]:
# ============================================================
# ANNOTATIONS + BASELINE GT REPRESENTATION  (unchanged from Notebook 03)
# Baseline target = FIRST annotation record per file (sorted by id).
# ============================================================

def load_annotations(path):
    with open(path) as f:
        coco = json.load(f)
    images_info = {im["id"]: im for im in coco["images"]}
    anns_by_image = defaultdict(list)
    for a in coco["annotations"]:
        anns_by_image[a["image_id"]].append(a)
    return coco, images_info, anns_by_image

COCO, images_info, anns_by_image = load_annotations(ANN_PATH)
print(f"JSON records: {len(images_info)} | annotations: {len(COCO['annotations'])}")

file_to_records = defaultdict(list)
for img_id, info in images_info.items():
    fname = Path(info["file_name"]).name
    if fname in name_to_path:
        file_to_records[fname].append(img_id)

missing = len(images_info) - sum(len(v) for v in file_to_records.values())
if missing:
    raise RuntimeError(f"{missing} JSON records reference files not on disk")

file_to_primary = {f: sorted(ids)[0] for f, ids in file_to_records.items()}
primary_ids = sorted(file_to_primary.values())
print(f"physical files: {len(file_to_primary)} | primary records: {len(primary_ids)}")

def ann_to_mask(ann, h, w):
    m = np.zeros((h, w), np.uint8)
    for poly in ann["segmentation"]:
        pts = np.asarray(poly, np.float64).reshape(-1, 2)
        cv2.fillPoly(m, [np.round(pts).astype(np.int32)], 1)
    return m.astype(bool)

def build_gt_masks(img_id):
    """Full-resolution instance masks for one record."""
    info = images_info[img_id]
    masks = [ann_to_mask(a, info["height"], info["width"])
             for a in anns_by_image[img_id]]
    masks = [m for m in masks if m.any()]
    if not masks:
        raise ValueError(f"Record {img_id} has no non-empty GT masks")
    return masks

def load_image(path):
    img = np.array(Image.open(path))
    if img.ndim == 3:
        img = img[..., :3].mean(axis=2)
    return img.astype(np.float32)

JSON records: 1154 | annotations: 8199
physical files: 707 | primary records: 707


In [5]:
# ============================================================
# OFFICIAL EVALUATOR REPRODUCTION
# Many-to-many matching: EVERY (pred, gt) pair with IoU > 0.5 (STRICT >)
# is a TP and contributes its IoU to the numerator.
# FP = pred with zero qualifying matches; FN = gt with zero.
# PQ = sum(TP IoUs) / (TP + 0.5 FP + 0.5 FN)
# Diagnostics (one-to-many / many-to-one) use overlap > 0.
#
# Only change vs Notebook 03: the pairwise IoU/Dice loop skips pairs whose
# bounding boxes do not intersect and only intersects the bbox overlap region.
# Results are mathematically identical (verified against the original
# implementation in the equivalence test below); it is just much faster.
# ============================================================

def _bbox(m):
    rows = np.flatnonzero(m.any(axis=1)); cols = np.flatnonzero(m.any(axis=0))
    if len(rows) == 0:
        return (0, 0, 0, 0)
    return (rows[0], rows[-1] + 1, cols[0], cols[-1] + 1)

def iou_and_dice_matrix(pred_masks, gt_masks):
    n_p, n_g = len(pred_masks), len(gt_masks)
    iou = np.zeros((n_p, n_g)); dce = np.zeros((n_p, n_g))
    pb = [_bbox(p) for p in pred_masks]; gb = [_bbox(g) for g in gt_masks]
    pa = [int(p.sum()) for p in pred_masks]; ga = [int(g.sum()) for g in gt_masks]
    for i, p in enumerate(pred_masks):
        py0, py1, px0, px1 = pb[i]
        for j, g in enumerate(gt_masks):
            gy0, gy1, gx0, gx1 = gb[j]
            y0, y1 = max(py0, gy0), min(py1, gy1)
            x0, x1 = max(px0, gx0), min(px1, gx1)
            if y0 >= y1 or x0 >= x1:
                continue
            inter = int(np.logical_and(p[y0:y1, x0:x1], g[y0:y1, x0:x1]).sum())
            if inter == 0:
                continue
            iou[i, j] = inter / (pa[i] + ga[j] - inter)
            dce[i, j] = 2 * inter / (pa[i] + ga[j])
    return iou, dce

def compute_official_pq(gt_masks, pred_masks):
    if len(gt_masks) == 0 and len(pred_masks) == 0:
        return {"PQ": 1.0, "TP": 0, "FP": 0, "FN": 0,
                "tp_ious": [], "tp_dices": [],
                "one_to_many": 0, "many_to_one": 0}
    iou, dce = iou_and_dice_matrix(pred_masks, gt_masks)
    hit = iou > 0.5                                   # STRICT >
    tp_ious = iou[hit].tolist()
    tp_dices = dce[hit].tolist()
    tp = len(tp_ious)
    fp = int((hit.sum(axis=1) == 0).sum()) if len(pred_masks) else 0
    fn = int((hit.sum(axis=0) == 0).sum()) if len(gt_masks) else 0
    denom = tp + 0.5 * fp + 0.5 * fn
    overlap = iou > 0
    one_to_many = int((overlap.sum(axis=0) > 1).sum())   # GT hit by multiple preds
    many_to_one = int((overlap.sum(axis=1) > 1).sum())   # pred hitting multiple GTs
    return {"PQ": (sum(tp_ious) / denom) if denom > 0 else 0.0,
            "TP": tp, "FP": fp, "FN": fn,
            "tp_ious": tp_ious, "tp_dices": tp_dices,
            "one_to_many": one_to_many, "many_to_one": many_to_one}

# ---------------- MANDATORY UNIT TESTS (same 7 as Notebook 03) ----------------
def _box(y0, y1, x0, x1, hw=64):
    m = np.zeros((hw, hw), bool); m[y0:y1, x0:x1] = True; return m

def run_metric_tests():
    results = []
    g = _box(10, 20, 10, 30)                       # 10x20 = 200 px

    r = compute_official_pq([g], [g.copy()])
    results.append(("T1 perfect -> PQ=1", abs(r["PQ"] - 1.0) < 1e-9))

    half = _box(10, 20, 10, 20)                    # IoU = 0.5 exactly
    r = compute_official_pq([g], [half])
    results.append(("T2 IoU==0.5 not TP", r["TP"] == 0 and r["FP"] == 1 and r["FN"] == 1))

    most = _box(10, 20, 10, 24)                    # IoU = 0.7
    r = compute_official_pq([g], [most])
    results.append(("T3 IoU>0.5 is TP", r["TP"] == 1 and r["FP"] == 0 and r["FN"] == 0))

    p1 = _box(10, 20, 10, 27); p2 = _box(10, 20, 12, 30)
    r = compute_official_pq([g], [p1, p2])
    results.append(("T4 1 GT + 2 preds -> TP=2 FP=0 FN=0",
                    r["TP"] == 2 and r["FP"] == 0 and r["FN"] == 0))

    g1 = _box(10, 20, 10, 30); g2 = _box(10, 20, 8, 32); pred = _box(10, 20, 9, 31)
    r = compute_official_pq([g1, g2], [pred])
    results.append(("T5 2 GT + 1 pred -> TP=2 FP=0 FN=0",
                    r["TP"] == 2 and r["FP"] == 0 and r["FN"] == 0))

    r = compute_official_pq([g], [])
    results.append(("T6 no preds -> FN=1, PQ=0", r["FN"] == 1 and r["PQ"] == 0.0))

    r = compute_official_pq([], [g])
    results.append(("T7 no GT -> FP=1, PQ=0", r["FP"] == 1 and r["PQ"] == 0.0))

    all_ok = True
    for name, ok in results:
        print(f"  {'PASS' if ok else 'FAIL'}  {name}")
        all_ok &= ok
    if not all_ok:
        raise AssertionError("Metric unit tests FAILED — do not proceed")
    print("All 7 metric tests passed.")

run_metric_tests()

# ---------------- EQUIVALENCE TEST: fast matrix == original Notebook-03 matrix ----------------
def _iou_and_dice_matrix_reference(pred_masks, gt_masks):
    """Verbatim original implementation from Notebook 03 (slow, full-frame)."""
    n_p, n_g = len(pred_masks), len(gt_masks)
    iou = np.zeros((n_p, n_g)); dce = np.zeros((n_p, n_g))
    for i, p in enumerate(pred_masks):
        ps = p.sum()
        for j, g in enumerate(gt_masks):
            inter = np.logical_and(p, g).sum()
            if inter == 0:
                continue
            gs = g.sum()
            iou[i, j] = inter / (ps + gs - inter)
            dce[i, j] = 2 * inter / (ps + gs)
    return iou, dce

def run_equivalence_test(n_trials=40, hw=96):
    rng = np.random.RandomState(0)
    def rand_mask():
        m = np.zeros((hw, hw), np.uint8)
        for _ in range(rng.randint(1, 4)):
            y, x = rng.randint(0, hw - 8, 2); h, w = rng.randint(2, 30, 2)
            m[y:y + h, x:x + w] = 1
        return m.astype(bool)
    for _ in range(n_trials):
        P = [rand_mask() for _ in range(rng.randint(0, 6))]
        G = [rand_mask() for _ in range(rng.randint(0, 6))]
        a = iou_and_dice_matrix(P, G); b = _iou_and_dice_matrix_reference(P, G)
        assert np.allclose(a[0], b[0]) and np.allclose(a[1], b[1]), "fast IoU/Dice differs from original"
    print(f"Equivalence test passed: fast evaluator == original on {n_trials} random cases.")

run_equivalence_test()

  PASS  T1 perfect -> PQ=1
  PASS  T2 IoU==0.5 not TP
  PASS  T3 IoU>0.5 is TP
  PASS  T4 1 GT + 2 preds -> TP=2 FP=0 FN=0
  PASS  T5 2 GT + 1 pred -> TP=2 FP=0 FN=0
  PASS  T6 no preds -> FN=1, PQ=0
  PASS  T7 no GT -> FP=1, PQ=0
All 7 metric tests passed.
Equivalence test passed: fast evaluator == original on 40 random cases.


In [6]:
# ============================================================
# SPLIT — date-grouped 80/20 (Notebook 02 A3). Same function, same seed as Notebook 03.
# ============================================================

FN_RE = re.compile(r"^(\d{4})(\d{2})(\d{2})\d{6}([A-Za-z])h\.", re.I)

def create_split(primary_ids, val_frac=0.2, seed=SEED):
    rows = []
    for img_id in primary_ids:
        fname = Path(images_info[img_id]["file_name"]).name
        m = FN_RE.match(fname)
        if not m:
            raise ValueError(f"Unparseable filename: {fname}")
        rows.append({"img_id": img_id, "file": fname,
                     "date": f"{m[1]}-{m[2]}-{m[3]}", "station": m[4].upper()})
    df = pd.DataFrame(rows)
    dates = sorted(df["date"].unique())
    rng = random.Random(seed); rng.shuffle(dates)
    n_val_dates = int(round(len(dates) * val_frac))
    val_dates = set(dates[:n_val_dates])
    df["split"] = df["date"].map(lambda d: "val" if d in val_dates else "train")
    return df

split_df = create_split(primary_ids)
train_ids = split_df.loc[split_df.split == "train", "img_id"].tolist()
val_ids   = split_df.loc[split_df.split == "val", "img_id"].tolist()

if DEBUG_MODE:                       # smoke-test shortcut — must be OFF for Baseline-01
    train_ids = train_ids[:CFG["debug_n_train"]]
    val_ids   = val_ids[:CFG["debug_n_val"]]

SPLIT_HASH = hashlib.sha1(",".join(sorted(
    split_df.loc[split_df.split == "val", "file"])).encode()).hexdigest()[:12]

split_df.to_csv(OUT / "split_metadata.csv", index=False)
print(f"train: {len(train_ids)} | val: {len(val_ids)} (date-grouped) | split hash: {SPLIT_HASH}")
assert not set(split_df[split_df.split == "train"].date) & \
           set(split_df[split_df.split == "val"].date)

train: 568 | val: 139 (date-grouped) | split hash: 0cc2d729a5b1


In [7]:
# ============================================================
# DATASET — fixed-size tensors. Preprocessing and augmentation (h/v flips only)
# are IDENTICAL to Notebook 03.
# ============================================================

class FilamentDataset(Dataset):
    def __init__(self, record_ids, resolution, augment=False):
        self.ids = record_ids
        self.res = resolution
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        info = images_info[img_id]
        fname = Path(info["file_name"]).name
        img = load_image(name_to_path[fname])
        h, w = img.shape

        gt = np.zeros((h, w), bool)
        for a in anns_by_image[img_id]:
            gt |= ann_to_mask(a, h, w)

        img = cv2.resize(img, (self.res, self.res), interpolation=cv2.INTER_AREA)
        gt = cv2.resize(gt.astype(np.uint8), (self.res, self.res),
                        interpolation=cv2.INTER_NEAREST).astype(bool)

        lo, hi = np.percentile(img, [1, 99])
        img = np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)

        if self.augment:
            if random.random() < 0.5:
                img = np.fliplr(img).copy(); gt = np.fliplr(gt).copy()
            if random.random() < 0.5:
                img = np.flipud(img).copy(); gt = np.flipud(gt).copy()

        return {"image": torch.from_numpy(img)[None],
                "mask": torch.from_numpy(gt.astype(np.float32))[None],
                "img_id": img_id}

def _worker_init(_):
    cv2.setNumThreads(0)

train_ds = FilamentDataset(train_ids, CFG["train_resolution"], augment=True)
val_ds   = FilamentDataset(val_ids, CFG["train_resolution"], augment=False)

# generator is re-seeded every epoch (SEED + epoch) so shuffling/augmentation is reproducible
# and identical whether or not the run was resumed.
train_gen = torch.Generator(); train_gen.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=True, drop_last=True,
                          generator=train_gen, worker_init_fn=_worker_init)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                        num_workers=CFG["num_workers"], pin_memory=True,
                        worker_init_fn=_worker_init)

# sample check WITHOUT creating DataLoader iterators (avoids the dangling-worker noise seen in Notebook 03)
_s = train_ds[0]
print("sample check:", tuple(_s["image"].shape), tuple(_s["mask"].shape),
      "| train batches/epoch:", len(train_loader), "| val images:", len(val_ds))
del _s

sample check: (1, 1024, 1024) (1, 1024, 1024) | train batches/epoch: 142 | val images: 139


In [8]:
# ============================================================
# MODEL — plain U-Net (base 32), UNCHANGED.  Loss — BCE + soft Dice, UNCHANGED
# (logits are cast to float32 inside the loss so AMP cannot affect the loss math).
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, cin=1, base=32):
        super().__init__()
        self.e1, self.e2 = ConvBlock(cin, base), ConvBlock(base, base*2)
        self.e3, self.e4 = ConvBlock(base*2, base*4), ConvBlock(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.mid = ConvBlock(base*8, base*16)
        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, 2); self.d4 = ConvBlock(base*16, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2);  self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2);  self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2);    self.d1 = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        m = self.mid(self.pool(e4))
        d = self.d4(torch.cat([self.u4(m), e4], 1))
        d = self.d3(torch.cat([self.u3(d), e3], 1))
        d = self.d2(torch.cat([self.u2(d), e2], 1))
        d = self.d1(torch.cat([self.u1(d), e1], 1))
        return self.out(d)

def build_model():
    return UNet(1, 32).to(DEVICE)

def build_optimizer(model):
    return torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

def bce_dice_loss(logits, target, eps=1.0):
    logits = logits.float()
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    inter = (prob * target).sum(dim=(2, 3))
    dice = (2 * inter + eps) / (prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + eps)
    return bce + (1 - dice).mean()

USE_AMP = bool(CFG["use_amp"]) and DEVICE.type == "cuda"

def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP)

def build_scaler():
    return torch.amp.GradScaler("cuda", enabled=USE_AMP)

EXPECTED_PARAMS = 7_765_409      # parameter count reported by the Notebook 03 smoke test
_tmp_model = build_model()
N_PARAMS = sum(p.numel() for p in _tmp_model.parameters())
del _tmp_model
print(f"parameters: {N_PARAMS:,} | AMP: {USE_AMP}")

parameters: 7,765,409 | AMP: True


In [9]:
# ============================================================
# INFERENCE + VALIDATION UNDER THE OFFICIAL METRIC (per-image, full 2048)
# Inference for PQ is done in FP32 (identical to Notebook 03).
# ============================================================

@torch.no_grad()
def predict_image(model, img_path, resolution):
    """Returns full-2048 probability map."""
    img = load_image(img_path)
    h, w = img.shape
    small = cv2.resize(img, (resolution, resolution), interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(small, [1, 99])
    small = np.clip((small - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)
    x = torch.from_numpy(small)[None, None].to(DEVICE)
    prob = torch.sigmoid(model(x))[0, 0].cpu().numpy()
    return cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)

def extract_instances(prob, threshold, min_px=0):
    """Probability map -> list of boolean instance masks via connected components."""
    binary = (prob > threshold).astype(np.uint8)
    n, labels = cv2.connectedComponents(binary, connectivity=8)
    out = []
    for k in range(1, n):
        m = labels == k
        if m.sum() > min_px:
            out.append(m)
    return out

def evaluate_validation(model, ids, threshold, desc="validation"):
    """Official-metric evaluation over ALL images in `ids`."""
    model.eval()
    rows, all_tp_ious, all_tp_dices = [], [], []
    agg = Counter()
    for img_id in tqdm(ids, desc=desc, leave=False):
        fname = Path(images_info[img_id]["file_name"]).name
        gt = build_gt_masks(img_id)
        prob = predict_image(model, name_to_path[fname], CFG["train_resolution"])
        preds = extract_instances(prob, threshold, CFG["min_instance_px"])
        r = compute_official_pq(gt, preds)
        all_tp_ious += r["tp_ious"]; all_tp_dices += r["tp_dices"]
        for k in ["TP", "FP", "FN", "one_to_many", "many_to_one"]:
            agg[k] += r[k]
        rows.append({"img_id": img_id, "file": fname,
                     "n_gt": len(gt), "n_pred": len(preds),
                     "PQ": r["PQ"], "TP": r["TP"], "FP": r["FP"], "FN": r["FN"],
                     "one_to_many": r["one_to_many"], "many_to_one": r["many_to_one"],
                     "mean_tp_iou": float(np.mean(r["tp_ious"])) if r["tp_ious"] else np.nan,
                     "mean_tp_dice": float(np.mean(r["tp_dices"])) if r["tp_dices"] else np.nan})
    per_image = pd.DataFrame(rows)
    denom = agg["TP"] + 0.5 * agg["FP"] + 0.5 * agg["FN"]
    overall = {
        "PQ_pooled": float(sum(all_tp_ious) / denom) if denom else 0.0,
        "PQ_mean_per_image": float(per_image["PQ"].mean()),
        "TP": int(agg["TP"]), "FP": int(agg["FP"]), "FN": int(agg["FN"]),
        "mean_tp_iou": float(np.mean(all_tp_ious)) if all_tp_ious else 0.0,
        "mean_tp_dice": float(np.mean(all_tp_dices)) if all_tp_dices else 0.0,
        "one_to_many": int(agg["one_to_many"]), "many_to_one": int(agg["many_to_one"]),
        "n_images_evaluated": int(len(per_image)), "threshold": float(threshold),
    }
    return per_image, overall, all_tp_ious, all_tp_dices

In [10]:
# ============================================================
# CONSISTENCY WITH BASELINE-01 + HELPERS
# ============================================================
prev_split = pd.read_csv(PREV / "split_metadata.csv")
assert set(prev_split.loc[prev_split.split == "val", "file"]) == set(split_df.loc[split_df.split == "val", "file"]), \
    "Validation split differs from Baseline-01 — stop."
print("Validation split identical to Baseline-01:", len(val_ids), "files | hash", SPLIT_HASH)

hist_path, meta_path = PREV / "training_history.csv", PREV / "baseline_best_meta.json"
if hist_path.exists():
    n_ep = len(pd.read_csv(hist_path)); print(f"training_history.csv: {n_ep} epochs")
    if n_ep < CFG["epochs"]:
        raise RuntimeError(f"Baseline-01 input is INCOMPLETE ({n_ep}/{CFG['epochs']} epochs). Attach the finished version.")
if meta_path.exists():
    meta = json.loads(meta_path.read_text()); print(f"baseline_best.pth = epoch {meta['best_epoch']} (val loss {meta['best_val_loss']:.4f})")
prev_res = pd.read_csv(PREV / "baseline_results.csv").iloc[0] if (PREV / "baseline_results.csv").exists() else None
if prev_res is not None:
    print(f"Baseline-01 saved result: PQ pooled {prev_res['PQ_pooled']:.4f} | mean/img {prev_res['PQ_mean_per_image']:.4f}")

def summarize(items):
    tp = sum(r["TP"] for r in items); fp = sum(r["FP"] for r in items); fn = sum(r["FN"] for r in items)
    ious = [x for r in items for x in r["tp_ious"]]
    denom = tp + 0.5 * fp + 0.5 * fn
    return {"PQ_pooled": (sum(ious) / denom) if denom else 0.0,
            "PQ_mean_per_image": float(np.mean([r["PQ"] for r in items])) if items else 0.0,
            "TP": tp, "FP": fp, "FN": fn, "mean_tp_iou": float(np.mean(ious)) if ious else 0.0}

vsplit = split_df[split_df.img_id.isin(val_ids)]
date_half = {d: "AB"[i % 2] for i, d in enumerate(sorted(vsplit.date.unique()))}
img_half = {r.img_id: date_half[r.date] for r in vsplit.itertuples()}

def load_model(path):
    ck = torch.load(path, map_location="cpu", weights_only=True)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    m = build_model(); m.load_state_dict(sd); m.eval()
    return m

def recalibrate_bn(model, n_batches=60):
    """Re-estimate BatchNorm running statistics on ~240 training images (weights untouched)."""
    m = copy.deepcopy(model)
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.reset_running_stats(); mod.momentum = None
    g = torch.Generator(); g.manual_seed(SEED)
    loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"],
                        drop_last=True, generator=g, worker_init_fn=_worker_init)
    it = iter(loader); m.train()
    with torch.no_grad():
        for _ in range(n_batches):
            b = next(it)
            with autocast_ctx():
                m(b["image"].to(DEVICE))
    del it; m.eval()
    return m

Validation split identical to Baseline-01: 139 files | hash 0cc2d729a5b1
training_history.csv: 50 epochs
baseline_best.pth = epoch 42 (val loss 0.3619)
Baseline-01 saved result: PQ pooled 0.2881 | mean/img 0.2974


In [11]:
# ============================================================
# BUILD PROBABILITY SOURCES (all cached at model resolution, float32)
#   best_asis      : Baseline-01 best checkpoint exactly as saved (reference)
#   <m>_bn         : model m with BN statistics re-estimated, plain prediction
#   <m>_bn_tta     : same + horizontal/vertical/both flip TTA (4 views averaged)
#   ens_bn(_tta)   : average over the models best/30/40/50 (snapshot ensemble)
# ============================================================
MODEL_SPECS = [("best", PREV / "baseline_best.pth")]
for ep in (30, 40, 50):
    p = PREV / f"checkpoint_epoch_{ep:03d}.pth"
    if p.exists():
        MODEL_SPECS.append((f"e{ep}", p))
print("models:", [n for n, _ in MODEL_SPECS])

@torch.no_grad()
def predict_views(model, img_path, resolution, tta=True):
    img = load_image(img_path)
    small = cv2.resize(img, (resolution, resolution), interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(small, [1, 99])
    small = np.clip((small - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)
    x = torch.from_numpy(small)[None, None].to(DEVICE)
    views = [None, [3], [2], [2, 3]] if tta else [None]
    outs = []
    for dims in views:
        xi = x if dims is None else torch.flip(x, dims)
        p = torch.sigmoid(model(xi))
        if dims is not None:
            p = torch.flip(p, dims)
        outs.append(p[0, 0].cpu().numpy())
    return outs[0], np.mean(outs, axis=0), img.shape

PROBS = defaultdict(dict)
for name, path in MODEL_SPECS:
    m0 = load_model(path)
    if name == "best":
        for img_id in tqdm(val_ids, desc="best (as saved)"):
            fn = Path(images_info[img_id]["file_name"]).name
            plain, _, shp = predict_views(m0, name_to_path[fn], CFG["train_resolution"], tta=False)
            PROBS["best_asis"][img_id] = (plain, shp)
    mr = recalibrate_bn(m0)
    for img_id in tqdm(val_ids, desc=f"{name}: BN re-estimated (+TTA)"):
        fn = Path(images_info[img_id]["file_name"]).name
        plain, tta_p, shp = predict_views(mr, name_to_path[fn], CFG["train_resolution"], tta=True)
        PROBS[f"{name}_bn"][img_id] = (plain, shp)
        PROBS[f"{name}_bn_tta"][img_id] = (tta_p, shp)
    del m0, mr; gc.collect(); torch.cuda.empty_cache()

mnames = [n for n, _ in MODEL_SPECS]
for suffix in ("bn", "bn_tta"):
    for img_id in val_ids:
        arrs = [PROBS[f"{n}_{suffix}"][img_id][0] for n in mnames]
        PROBS[f"ens_{suffix}"][img_id] = (np.mean(arrs, axis=0), PROBS[f"{mnames[0]}_{suffix}"][img_id][1])
SOURCES = list(PROBS.keys())
print("sources:", SOURCES)

def full_prob(src, img_id):
    p, (h, w) = PROBS[src][img_id]
    return cv2.resize(p, (w, h), interpolation=cv2.INTER_LINEAR)

models: ['best', 'e30', 'e40', 'e50']


best (as saved):   0%|          | 0/139 [00:00<?, ?it/s]

best: BN re-estimated (+TTA):   0%|          | 0/139 [00:00<?, ?it/s]

e30: BN re-estimated (+TTA):   0%|          | 0/139 [00:00<?, ?it/s]

e40: BN re-estimated (+TTA):   0%|          | 0/139 [00:00<?, ?it/s]

e50: BN re-estimated (+TTA):   0%|          | 0/139 [00:00<?, ?it/s]

sources: ['best_asis', 'best_bn', 'best_bn_tta', 'e30_bn', 'e30_bn_tta', 'e40_bn', 'e40_bn_tta', 'e50_bn', 'e50_bn_tta', 'ens_bn', 'ens_bn_tta']


In [ ]:
# ============================================================
# A/B/C — which inference-time ingredient helps?  (post-processing held fixed at two settings)
#   setting 1: threshold 0.5, no size filter  (= Baseline-01 post-processing)
#   setting 2: threshold 0.5, min object 200 px (removes the ~400 tiny false positives; GT has almost none that small)
# ============================================================
SETTINGS = [(0.5, 0), (0.5, 200)]
RS = {(s, st): [] for s in SOURCES for st in SETTINGS}
for img_id in tqdm(val_ids, desc="evaluate sources"):
    gt = build_gt_masks(img_id)
    for s in SOURCES:
        prob = full_prob(s, img_id)
        n, labels = cv2.connectedComponents((prob > 0.5).astype(np.uint8), connectivity=8)
        areas = np.bincount(labels.ravel(), minlength=n)
        comps = [(int(areas[k]), labels == k) for k in range(1, n)]
        for st in SETTINGS:
            RS[(s, st)].append((img_id, compute_official_pq(gt, [mk for a, mk in comps if a > st[1]])))
        del comps

rows = []
for (s, st), items in RS.items():
    allr = [r for _, r in items]
    A = [r for i, r in items if img_half[i] == "A"]; B = [r for i, r in items if img_half[i] == "B"]
    sm = summarize(allr)
    rows.append({"source": s, "threshold": st[0], "min_px": st[1], "PQ_all": sm["PQ_pooled"],
                 "PQ_mean_per_image": sm["PQ_mean_per_image"], "PQ_A": summarize(A)["PQ_pooled"],
                 "PQ_B": summarize(B)["PQ_pooled"], "TP": sm["TP"], "FP": sm["FP"], "FN": sm["FN"]})
src_df = pd.DataFrame(rows)
src_df.to_csv(OUT / "sources_table.csv", index=False)
print("Pooled PQ by probability source (139 validation images):")
print(src_df.pivot(index="source", columns="min_px", values="PQ_all").round(4).to_string())
base_row = src_df[(src_df.source == "best_asis") & (src_df.min_px == 0)].iloc[0]
print(f"\nReference (Baseline-01 as saved, thr 0.5, min 0): PQ {base_row.PQ_all:.4f}")
SRC_PICK = src_df[src_df.min_px == 200].sort_values("PQ_all", ascending=False).iloc[0].source
print(f"Source with the best PQ at min_px=200: {SRC_PICK}  (used for the extended sweep below)")

evaluate sources:   0%|          | 0/139 [00:00<?, ?it/s]

In [13]:
# ============================================================
# D — EXTENDED POST-PROCESSING SWEEP on the chosen source (split-half protocol)
# Previous sweep's best settings were at the edge of the grid, so the grid now extends further.
# ============================================================
THRS2 = [0.5, 0.6, 0.7, 0.8]
MINS2 = [200, 400, 800, 1500]
CLOSE2 = [0, 10, 20]

RES2 = {(t, m_, c): [] for t in THRS2 for m_ in MINS2 for c in CLOSE2}
for img_id in tqdm(val_ids, desc=f"extended sweep on {SRC_PICK}"):
    prob = full_prob(SRC_PICK, img_id); gt = build_gt_masks(img_id)
    for t in THRS2:
        binary = (prob > t).astype(np.uint8)
        for c in CLOSE2:
            bm = binary if c == 0 else cv2.morphologyEx(
                binary, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * c + 1, 2 * c + 1)))
            n, labels = cv2.connectedComponents(bm, connectivity=8)
            areas = np.bincount(labels.ravel(), minlength=n)
            comps = [(int(areas[k]), labels == k) for k in range(1, n)]
            for m_ in MINS2:
                RES2[(t, m_, c)].append((img_id, compute_official_pq(gt, [mk for a, mk in comps if a > m_])))
            del comps

rows = []
for (t, m_, c), items in RES2.items():
    allr = [r for _, r in items]
    A = [r for i, r in items if img_half[i] == "A"]; B = [r for i, r in items if img_half[i] == "B"]
    sm = summarize(allr)
    rows.append({"threshold": t, "min_px": m_, "closing_r": c, "PQ_all": sm["PQ_pooled"], "PQ_A": summarize(A)["PQ_pooled"],
                 "PQ_B": summarize(B)["PQ_pooled"], "TP": sm["TP"], "FP": sm["FP"], "FN": sm["FN"]})
sw2 = pd.DataFrame(rows).sort_values("PQ_all", ascending=False).reset_index(drop=True)
sw2.to_csv(OUT / "extended_sweep.csv", index=False)
print("Top 10 settings by PQ on all 139 images:"); print(sw2.head(10).round(4).to_string(index=False))

edge = sw2.iloc[0]
on_edge = (edge.threshold in (min(THRS2), max(THRS2))) or (edge.min_px in (min(MINS2), max(MINS2))) or (edge.closing_r in (min(CLOSE2), max(CLOSE2)))
print("\nBest setting sits on the edge of the grid ->", "YES (optimum may lie beyond; extend again)" if on_edge else "no (interior optimum)")

bestA = sw2.loc[sw2.PQ_A.idxmax()]; bestB = sw2.loc[sw2.PQ_B.idxmax()]
refA = src_df[(src_df.source == "best_asis") & (src_df.min_px == 0)].iloc[0]
print(f"\nBaseline-01 on halves: A {refA.PQ_A:.4f} | B {refA.PQ_B:.4f}")
print(f"Chosen on A (thr {bestA.threshold}, min {int(bestA.min_px)}, close {int(bestA.closing_r)}) -> held-out B {bestA.PQ_B:.4f}  (vs baseline {refA.PQ_B:.4f}: {bestA.PQ_B - refA.PQ_B:+.4f})")
print(f"Chosen on B (thr {bestB.threshold}, min {int(bestB.min_px)}, close {int(bestB.closing_r)}) -> held-out A {bestB.PQ_A:.4f}  (vs baseline {refA.PQ_A:.4f}: {bestB.PQ_A - refA.PQ_A:+.4f})")
HELD_OUT_PQ = float(np.mean([bestA.PQ_B, bestB.PQ_A])); HELD_OUT_BASE = float(np.mean([refA.PQ_B, refA.PQ_A]))
print(f"Held-out PQ of the whole recipe (source + post-processing): {HELD_OUT_PQ:.4f} vs Baseline-01 {HELD_OUT_BASE:.4f}  ({HELD_OUT_PQ - HELD_OUT_BASE:+.4f})")

RECIPE = {"source": SRC_PICK, "threshold": float(edge.threshold), "min_px": int(edge.min_px), "closing_r": int(edge.closing_r),
          "PQ_all": float(edge.PQ_all), "held_out_PQ_estimate": HELD_OUT_PQ, "baseline_PQ_all": float(base_row.PQ_all)}
(OUT / "recipe.json").write_text(json.dumps(RECIPE, indent=2)); print("\nrecipe (diagnostic):", RECIPE)

Top 10 settings by PQ on all 139 images:
 threshold  min_px  closing_r  PQ_all   PQ_A   PQ_B  TP  FP  FN
       0.5     400         10  0.3720 0.3804 0.3628 576 489 432
       0.6     400         10  0.3713 0.3779 0.3642 562 466 446
       0.7     400         10  0.3674 0.3806 0.3532 543 444 465
       0.6     400         20  0.3669 0.3783 0.3546 557 458 451
       0.5     400         20  0.3641 0.3757 0.3516 566 487 442
       0.5     400          0  0.3633 0.3698 0.3562 562 520 446
       0.7     400         20  0.3628 0.3744 0.3502 541 446 467
       0.6     400          0  0.3615 0.3700 0.3523 547 491 461
       0.5     200         10  0.3581 0.3638 0.3520 612 665 396
       0.6     200         10  0.3571 0.3603 0.3536 599 642 409

Best setting sits on the edge of the grid -> YES (optimum may lie beyond; extend again)

Baseline-01 on halves: A 0.2927 | B 0.2833
Chosen on A (thr 0.7, min 400, close 10) -> held-out B 0.3532  (vs baseline 0.2833: +0.0700)
Chosen on B (thr 0.6, min 400

In [14]:
# ============================================================
# E — POOLED-ANNOTATOR GROUND TRUTH
# Same evaluator, but GT = instances from ALL annotation records of the image (not only the primary record).
# Exploratory: which GT definition the leaderboard uses must be confirmed from the competition's evaluation description.
# ============================================================
def get_preds(src, img_id, thr, min_px, close_r):
    prob = full_prob(src, img_id)
    binary = (prob > thr).astype(np.uint8)
    if close_r:
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE,
                                  cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * close_r + 1, 2 * close_r + 1)))
    n, labels = cv2.connectedComponents(binary, connectivity=8)
    areas = np.bincount(labels.ravel(), minlength=n)
    return [labels == k for k in range(1, n) if areas[k] > min_px]

CONFIGS = {"Baseline-01 (as saved, thr 0.5, no filter)": ("best_asis", 0.5, 0, 0),
           f"Recipe ({RECIPE['source']}, thr {RECIPE['threshold']}, min {RECIPE['min_px']}, close {RECIPE['closing_r']})":
               (RECIPE["source"], RECIPE["threshold"], RECIPE["min_px"], RECIPE["closing_r"])}
ER = {(c, g): [] for c in CONFIGS for g in ("primary", "pooled")}
n_multi = 0
for img_id in tqdm(val_ids, desc="pooled-GT evaluation"):
    fname = Path(images_info[img_id]["file_name"]).name
    gt_primary = build_gt_masks(img_id)
    gt_pooled = list(gt_primary)
    for rk in sorted(file_to_records[fname])[1:]:
        try:
            gt_pooled += build_gt_masks(rk)
        except ValueError:
            pass
    n_multi += int(len(gt_pooled) > len(gt_primary))
    for c, (s, t, m_, cr) in CONFIGS.items():
        preds = get_preds(s, img_id, t, m_, cr)
        ER[(c, "primary")].append(compute_official_pq(gt_primary, preds))
        ER[(c, "pooled")].append(compute_official_pq(gt_pooled, preds))

rows = []
for (c, g), items in ER.items():
    sm = summarize(items)
    rows.append({"config": c, "ground_truth": g, **{k: round(v, 4) if isinstance(v, float) else v for k, v in sm.items()}})
pool_df = pd.DataFrame(rows); pool_df.to_csv(OUT / "pooled_gt.csv", index=False)
print(f"images with extra annotation records: {n_multi} of {len(val_ids)}")
print(pool_df.to_string(index=False))

pooled-GT evaluation:   0%|          | 0/139 [00:00<?, ?it/s]

images with extra annotation records: 59 of 139
                                         config ground_truth  PQ_pooled  PQ_mean_per_image  TP   FP  FN  mean_tp_iou
     Baseline-01 (as saved, thr 0.5, no filter)      primary     0.2881             0.2974 611 1201 397       0.6649
     Baseline-01 (as saved, thr 0.5, no filter)       pooled     0.3565             0.3443 988 1116 619       0.6695
Recipe (e50_bn_tta, thr 0.5, min 400, close 10)      primary     0.3720             0.3746 576  489 432       0.6694
Recipe (e50_bn_tta, thr 0.5, min 400, close 10)       pooled     0.4224             0.4101 929  421 678       0.6723


In [15]:
# ============================================================
# SUMMARY — copy this block back to Claude
# ============================================================
L = []; add = L.append
add("NOTEBOOK 06 SUMMARY (no retraining; validation = 139 images)")
add("-" * 70)
add(f"Reference Baseline-01 (best_asis, thr 0.5, min 0): PQ {base_row.PQ_all:.4f}")
add("PQ by source at min_px 0 / 200:")
for s in SOURCES:
    a = src_df[(src_df.source == s) & (src_df.min_px == 0)].iloc[0].PQ_all; b = src_df[(src_df.source == s) & (src_df.min_px == 200)].iloc[0].PQ_all
    add(f"   {s:<14} {a:.4f} / {b:.4f}")
add(f"Source used for the extended sweep: {SRC_PICK}")
add("Top 5 extended-sweep settings (thr, min_px, closing, PQ_all, PQ_A, PQ_B):")
for r in sw2.head(5).itertuples():
    add(f"   {r.threshold}, {int(r.min_px)}, {int(r.closing_r)}: {r.PQ_all:.4f}, {r.PQ_A:.4f}, {r.PQ_B:.4f}   (TP {r.TP} FP {r.FP} FN {r.FN})")
add(f"Best setting on edge of grid: {'YES' if on_edge else 'no'}")
add(f"Held-out PQ of recipe: {HELD_OUT_PQ:.4f} vs Baseline-01 {HELD_OUT_BASE:.4f} ({HELD_OUT_PQ - HELD_OUT_BASE:+.4f})")
add("Pooled-annotator GT (config | GT | PQ pooled | TP FP FN):")
for r in pool_df.itertuples():
    add(f"   {r.config[:48]:<48} | {r.ground_truth:<7} | {r.PQ_pooled:.4f} | {r.TP} {r.FP} {r.FN}")
add("CAVEAT: diagnostics on the validation set that also chose the checkpoint; source/recipe selection adds optimism.")
rep = "\n".join(L); print(rep); (OUT / "notebook06_report.txt").write_text(rep)
print("\nSaved to", OUT, ":", sorted(p.name for p in OUT.iterdir()))

NOTEBOOK 06 SUMMARY (no retraining; validation = 139 images)
----------------------------------------------------------------------
Reference Baseline-01 (best_asis, thr 0.5, min 0): PQ 0.2881
PQ by source at min_px 0 / 200:
   best_asis      0.2881 / 0.3350
   best_bn        0.2751 / 0.3282
   best_bn_tta    0.2911 / 0.3422
   e30_bn         0.2631 / 0.3118
   e30_bn_tta     0.2665 / 0.3112
   e40_bn         0.2702 / 0.3182
   e40_bn_tta     0.2781 / 0.3275
   e50_bn         0.2953 / 0.3397
   e50_bn_tta     0.3044 / 0.3507
   ens_bn         0.2932 / 0.3417
   ens_bn_tta     0.2986 / 0.3407
Source used for the extended sweep: e50_bn_tta
Top 5 extended-sweep settings (thr, min_px, closing, PQ_all, PQ_A, PQ_B):
   0.5, 400, 10: 0.3720, 0.3804, 0.3628   (TP 576 FP 489 FN 432)
   0.6, 400, 10: 0.3713, 0.3779, 0.3642   (TP 562 FP 466 FN 446)
   0.7, 400, 10: 0.3674, 0.3806, 0.3532   (TP 543 FP 444 FN 465)
   0.6, 400, 20: 0.3669, 0.3783, 0.3546   (TP 557 FP 458 FN 451)
   0.5, 400, 20: 0.3